# 07 · E9 PaySim external evaluation

The same pipeline on the fixed 300,000-row PaySim sample (seed 31415, the same rows as Steps 4-4E). As in v5, the AE stops on 10% held-out legitimate training rows here.

1. **Representation**: baseline (7 features: transaction type, log amount, step) against enhanced (adds six features from balances known *before* the transaction). Post-transaction balances are never used.
2. **PCA on/off** for the baseline representation (PCA-on is reused from part 1, so the pairs are matched).
3. **Assumed contamination** c on the baseline scores, with no retraining.
4. **Diagnostics** by transaction type and amount band for seed 42.

The PaySim test data were examined before the enhanced representation was designed (v5 Step 4D), so part 1 is exploratory, not a confirmatory test.

| | |
|---|---|
| Trains models | yes: 5 seeds × 2 representations + 5 PCA-off fits |
| Expected time | roughly 20 min on a GPU runtime |
| Writes | `results/E9_paysim/`: `representation_by_seed.csv`, `pca_ablation_by_seed.csv`, `contamination_by_seed.csv`, diagnostics, `_summary.csv` files, `scores/` |

In [ ]:
import os, sys

ROOT = os.environ.get("RAREGUARD_ROOT", "/content/drive/MyDrive/PhD/RAREGuard_v6")

if ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

sys.path.insert(0, os.path.join(ROOT, "code"))
import rareguard_lib as rg

try:
    from IPython.display import display
except ImportError:
    display = print

cfg = rg.load_config(ROOT)
env = rg.show_environment()

In [ ]:
import gc
import numpy as np, pandas as pd

EXP = "E9_paysim"
FORCE_RERUN = False
RUN_TRAINING = FORCE_RERUN or not rg.already_done(cfg, EXP, ["representation_by_seed.csv", "pca_ablation_by_seed.csv"])

pay = rg.load_paysim_sample(cfg)
y = pay["isFraud"].to_numpy(dtype=np.int64)
saved_rows = rg.proj(cfg, "data", "splits", "paysim_sample_rows.npy")
if os.path.exists(saved_rows):
    assert np.array_equal(np.load(saved_rows), pay["source_row"].to_numpy()), "PaySim sample differs from notebook 00"
REPS = {r: rg.paysim_features(pay, r) for r in ("baseline", "enhanced")}
ES = cfg["paysim"]["ae_early_stopping"]
W, C = cfg["fusion_weights"], cfg["contamination"]
print(f"{len(y):,} rows, {int(y.sum())} frauds | features: baseline {len(REPS['baseline'][1])}, enhanced {len(REPS['enhanced'][1])}")

## 1-2. Representation and PCA ablation

In [ ]:
def fit_and_score(Xr, names, tr, va, te, seed, use_pca=True):
    F = rg.prepare_features(Xr[tr], Xr[va], Xr[te], names, seed, cfg, use_pca=use_pca)
    scores, secs, info = rg.fit_detectors(F, y[tr], seed, cfg, ae_early_stopping=ES)
    val_s = rg.weighted_fusion({m: scores[m][1] for m in scores}, W)
    test_s = rg.weighted_fusion({m: scores[m][2] for m in scores}, W)
    return F, scores, val_s, test_s, sum(secs.values())

if RUN_TRAINING:
    run = rg.Run(cfg, EXP, "07_E9_paysim.ipynb", "PaySim representation, PCA ablation, contamination, diagnostics")
    rep_rows, pca_rows = [], []
    for seed in cfg["seeds"]:
        tr, va, te = rg.get_split(cfg, "paysim", y, seed)
        for rep, (Xr, names) in REPS.items():
            F, scores, val_s, test_s, secs = fit_and_score(Xr, names, tr, va, te, seed)
            m = rg.evaluate(y[te], test_s, rg.percentile_threshold(val_s, C), val_s)
            row = {"Seed": seed, "Representation": rep.capitalize(), "Feature_Count": len(names),
                   "Selected_Features": "; ".join(F["selected"]), "Model_Dims": F["dims"],
                   "Train_Time_s": secs, **m}
            rep_rows.append(row)
            if rep == "baseline":
                pca_rows.append({**row, "PCA": "On"})
            run.save_scores(f"paysim_{rep}_seed{seed}", y_val=y[va], y_test=y[te], val_idx=va, test_idx=te,
                            fused_val=val_s, fused_test=test_s)
            print(f"seed {seed:>4} | {rep:<8} | PR-AUC {m['PR_AUC']:.3f} | TP {m['TP']} FP {m['FP']}")

        Xr, names = REPS["baseline"]
        F, scores, val_s, test_s, secs = fit_and_score(Xr, names, tr, va, te, seed, use_pca=False)
        m = rg.evaluate(y[te], test_s, rg.percentile_threshold(val_s, C), val_s)
        pca_rows.append({"Seed": seed, "Representation": "Baseline", "Feature_Count": len(names),
                         "Selected_Features": "; ".join(F["selected"]), "Model_Dims": F["dims"],
                         "Train_Time_s": secs, "PCA": "Off", **m})
        print(f"seed {seed:>4} | baseline, PCA off | PR-AUC {m['PR_AUC']:.3f}")
        gc.collect()

    rep_df, pca_df = pd.DataFrame(rep_rows), pd.DataFrame(pca_rows)
    run.save_csv(rep_df, "representation_by_seed.csv")
    run.save_csv(rg.mean_sd(rep_df, ["Representation"], rg.METRICS + ["TP", "FP", "Test_Alert_Rate"]), "representation_summary.csv")
    run.save_csv(pca_df, "pca_ablation_by_seed.csv")
    run.save_csv(rg.mean_sd(pca_df, ["PCA"], rg.METRICS + ["TP", "FP"]), "pca_ablation_summary.csv")

## 3-4. Assumed contamination and diagnostics (no training)

In [ ]:
if RUN_TRAINING:
    c_rows = []
    for seed in cfg["seeds"]:
        z = rg.load_scores(cfg, EXP, f"paysim_baseline_seed{seed}")
        for c in cfg["e5_contamination_grid"]:
            thr = rg.percentile_threshold(z["fused_val"], c)
            c_rows.append({"Seed": seed, "Contamination": c,
                           **rg.evaluate(z["y_test"], z["fused_test"], thr, z["fused_val"])})
    c_df = pd.DataFrame(c_rows)
    run.save_csv(c_df, "contamination_by_seed.csv")
    run.save_csv(rg.mean_sd(c_df, ["Contamination"], rg.METRICS + ["TP", "FP"]), "contamination_summary.csv")

    # Diagnostics for seed 42 only. Test labels describe outcomes; nothing here feeds back into the model.
    z = rg.load_scores(cfg, EXP, "paysim_baseline_seed42")
    held = pay.iloc[z["test_idx"]].copy()
    held["Score"] = z["fused_test"]
    held["Flagged"] = (z["fused_test"] >= rg.percentile_threshold(z["fused_val"], C)).astype(int)
    by_type = held.groupby("type").agg(N=("isFraud", "size"), Fraud=("isFraud", "sum"),
                                       Detected=("Flagged", lambda s: int((s & held.loc[s.index, "isFraud"]).sum())),
                                       False_Alerts=("Flagged", lambda s: int((s & (1 - held.loc[s.index, "isFraud"])).sum())),
                                       Mean_Score=("Score", "mean"))
    by_type["Recall"] = by_type.Detected / by_type.Fraud.replace(0, np.nan)
    run.save_csv(by_type.reset_index(), "detection_by_type_seed42.csv")

    bands = pd.cut(pay["amount"], [-np.inf, 100, 1e3, 1e4, 1e5, np.inf], labels=["<=100", "100-1k", "1k-10k", "10k-100k", ">100k"])
    by_amount = pay.groupby(bands, observed=False).agg(Transactions=("isFraud", "size"), Fraud=("isFraud", "sum"))
    by_amount["Fraud_Rate"] = by_amount.Fraud / by_amount.Transactions.replace(0, np.nan)
    run.save_csv(by_amount.rename_axis("Amount_Band").reset_index(), "fraud_by_amount.csv")
    run.save_csv(held.groupby("isFraud").Score.describe().reset_index(), "score_summary_seed42.csv")
    run.finish(notes="enhanced representation is exploratory (test set inspected in v5 Step 4D)")

In [ ]:
display(rg.read_result(cfg, EXP, "representation_summary.csv")[["Representation", "F1_mean", "PR_AUC_mean", "ROC_AUC_mean", "TP_mean", "FP_mean"]].round(4))
display(rg.read_result(cfg, EXP, "detection_by_type_seed42.csv"))